In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
tf.__version__

: 

In [3]:
hub_url = "https://www.kaggle.com/models/google/movinet/TensorFlow2/a5-stream-kinetics-600-classification/2"

encoder = hub.KerasLayer(hub_url, trainable=True)

# Define the image (video) input
image_input = tf.keras.layers.Input(
    shape=[None, None, None, 3],
    dtype=tf.float32,
    name='image')

# Define the state inputs, which is a dict that maps state names to tensors.
init_states_fn = encoder.resolved_object.signatures['init_states']
state_shapes = {
    name: ([s if s > 0 else None for s in state.shape], state.dtype)
    for name, state in init_states_fn(tf.constant([0, 0, 0, 0, 3])).items()
}


In [4]:
# states_input = {
#     name.replace('/', '_'): tf.keras.Input(shape[1:], dtype=dtype, name=name.replace('/', '_'))
#     for name, (shape, dtype) in state_shapes.items()
# }
states_input = {
    name: tf.keras.Input(shape[1:], dtype=dtype, name=name)
    for name, (shape, dtype) in state_shapes.items()
}

# The inputs to the model are the states and the video
inputs = {**states_input, 'image': image_input}

outputs = encoder(inputs)

In [5]:
model = tf.keras.Model(inputs, outputs, name='movinet')

# Create your example input here.
# Refer to the description or paper for recommended input shapes.
example_input = tf.ones([1, 8, 320, 320, 3])


frames = tf.split(example_input, example_input.shape[1], axis=1)

# Initialize the dict of states. All state tensors are initially zeros.
init_states = init_states_fn(tf.shape(example_input))

# Run the model prediction by looping over each frame.
states = init_states
predictions = []
for frame in frames:
  output, states = model({**states, 'image': frame})
  predictions.append(output)

# The video classification will simply be the last output of the model.
final_prediction = tf.argmax(predictions[-1], -1)

# # Alternatively, we can run the network on the entire input video.
# # The output should be effectively the same
# # (but it may differ a small amount due to floating point errors).
# non_streaming_output, _ = model({**init_states, 'image': example_input})
# non_streaming_prediction = tf.argmax(non_streaming_output, -1)

In [6]:
print(final_prediction)

tf.Tensor([50], shape=(1,), dtype=int64)


In [6]:
import os

def collect_video_paths(main_folder):
    """
    Collect all video files under main_folder/train and main_folder/val.
    Returns:
        video_dict = {
            'train': [(video_path, class_name), ...],
            'val': [(video_path, class_name), ...]
        }
    """
    video_dict = {'train': [], 'val': []}
    
    for split in ['train', 'val']:
        split_path = os.path.join(main_folder, split)
        if not os.path.exists(split_path):
            continue
        
        for class_name in sorted(os.listdir(split_path)):
            class_path = os.path.join(split_path, class_name)
            if not os.path.isdir(class_path):
                continue
            
            for file_name in os.listdir(class_path):
                if file_name.endswith('.mp4'):
                    full_path = os.path.join(class_path, file_name)
                    video_dict[split].append((full_path, class_name))
    
    return video_dict


In [7]:
video_dict = collect_video_paths("useful_videos")

In [8]:
video_dict['val']

[('useful_videos\\val\\drinking shots\\0fmNdKx4cdI_000003_000013.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\1BkmCXHttEQ_000012_000022.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\1GX9mW2PkL0_000000_000010.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\2ib0zTt3JGc_000002_000012.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\4ZUbtL-7Ogw_000045_000055.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\5XydIroPDng_000000_000010.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\7pIXtKfshBk_000015_000025.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\9GlFF7Fv_DQ_000005_000015.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\9xuCdsR5Krc_000051_000061.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\9ZYpzeRUFNg_000115_000125.mp4',
  'drinking shots'),
 ('useful_videos\\val\\drinking shots\\aEjonIp1yIk_000000_000010.mp4',
  'drinki

In [8]:
import os
import numpy as np
import imageio.v3 as iio
import tensorflow as tf

def preprocess_frame(frame, target_size=(172, 172)):
    """Resize and normalize a single frame."""
    frame = tf.image.resize(frame, target_size)
    frame = tf.cast(frame, tf.float32) / 255.0
    return frame.numpy()

def stream_video_frames(video_path, frames_per_step=8, stride=8, target_size=(172, 172)):
    """
    Stream frames from a video in steps of [1, frames_per_step, H, W, 3].
    Yields a single batch (1 video) per iteration.
    """
    frames = []
    reader = iio.imiter(video_path)

    for frame in reader:
        frames.append(preprocess_frame(frame, target_size))

        # When enough frames collected for a step, yield
        if len(frames) == frames_per_step:
            yield np.expand_dims(np.stack(frames, axis=0), axis=0)  # [1, frames_per_step, H, W, 3]
            frames = frames[stride:]  # slide by stride

    # Handle leftover frames (pad with last frame)
    if frames:
        while len(frames) < frames_per_step:
            frames.append(frames[-1])
        yield np.expand_dims(np.stack(frames, axis=0), axis=0)


In [9]:
def video_loader(video_dict, split='train', frames_per_step=8, stride=8, target_size=(172, 172)):
    """
    Generator that yields (video_path, class_name, frame_batches_generator).
    """
    for video_path, class_name in video_dict.get(split, []):
        yield video_path, class_name, stream_video_frames(video_path, frames_per_step, stride, target_size)


In [10]:
def load_labels_from_file(label_file_path):
    """
    Load labels from a txt file, where each line is a class name.
    Example line:
        assembling computer
        attending conference
        ...
    Returns:
        labels: list of class names
    """
    with open(label_file_path, 'r', encoding='utf-8') as f:
        labels = [line.strip() for line in f.readlines() if line.strip()]
    return labels
labels = load_labels_from_file("kinetics600_label_map.txt")

In [9]:
print("First 5 labels:", labels[:5])
len(labels)


First 5 labels: ['abseiling', 'acting in play', 'adjusting glasses', 'air drumming', 'alligator wrestling']


600

In [15]:
import json
from collections import defaultdict
import tensorflow as tf

def evaluate_videos_streaming_with_logs(
    video_dict,
    model,
    init_states_fn,
    labels,
    split='val',
    frames_per_step=8,
    stride=8,
    output_json='results.json'
):
    """
    Evaluate videos one-by-one using streaming inference and save logs with confidence.
    """
    class_to_idx = {cls: i for i, cls in enumerate(labels)}
    class_accuracy = defaultdict(lambda: {'correct': 0, 'total': 0})
    results_log = []

    for video_path, class_name, frame_batches in video_loader(
        video_dict, split, frames_per_step, stride
    ):
        # Initialize states for new video
        dummy_input = tf.zeros([1, frames_per_step, 172, 172, 3])
        states = init_states_fn(tf.shape(dummy_input))
        predictions = []

        # Stream video frame batches
        for batch in frame_batches:
            batch_tf = tf.convert_to_tensor(batch, dtype=tf.float32)
            output, states = model({**states, 'image': batch_tf})
            predictions.append(output)

        # Final prediction (last output)
        final_logits = predictions[-1]  # shape [1, num_classes]
        final_probs = tf.nn.softmax(final_logits, axis=-1)
        final_pred_idx = int(tf.argmax(final_probs, axis=-1).numpy()[0])
        confidence = float(tf.reduce_max(final_probs).numpy())  # max probability

        true_idx = class_to_idx[class_name]

        # Update accuracy stats
        class_accuracy[class_name]['total'] += 1
        if final_pred_idx == true_idx:
            class_accuracy[class_name]['correct'] += 1

        # Log the prediction with confidence
        results_log.append({
            "video": video_path,
            "true_label": class_name,
            "pred_label": labels[final_pred_idx],
            "pred_index": final_pred_idx,
            "true_index": int(true_idx),
            "confidence": confidence
        })

    # Save results to JSON
    with open(output_json, 'w') as f:
        json.dump(results_log, f, indent=4)

    # Print per-class accuracy
    print(f"\nResults for split: {split}")
    for cls, stats in class_accuracy.items():
        acc = stats['correct'] / stats['total'] * 100 if stats['total'] > 0 else 0
        print(f"Class {cls}: {acc:.2f}% ({stats['correct']}/{stats['total']})")

    return class_accuracy


In [16]:
acc = evaluate_videos_streaming_with_logs(video_dict, model, init_states_fn, labels)

OSError: Could not find a backend to open `useful_videos\val\drinking shots\0fmNdKx4cdI_000003_000013.mp4`` with iomode `r`.
Based on the extension, the following plugins might add capable backends:
  FFMPEG:  pip install imageio[ffmpeg]
  pyav:  pip install imageio[pyav]